# 第11課 - 代理人對代理人 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 甚麼是 A2A 協議？

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *代理卡* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A 定義了像 *已提交*、*進行中*、*已完成* 和 *失敗* 等狀態，給予協調者對被委派任務進度的完全可見性。

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## 建立專門化的旅遊代理人


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程的多代理協作

我們將三個代理連接成一個順序的工作流程，模仿 A2A 訊息傳遞：

1. **CurrencyExchangeAgent** 接收用戶請求並提供貨幣指引。
2. **ActivityPlannerAgent** 接收已豐富的上下文並新增活動建議。
3. **TravelManagerAgent** 將兩者輸入綜合成最終旅遊簡報。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生產環境中理解 A2A

在生產環境中，A2A 協議可以解鎖強大的跨服務場景：

| Capability | Description |
|---|---|
| **Cross-framework interop** | 使用一個框架建立的代理可以將任務委派給使用任何其他符合 A2A 的框架所建立的代理，從而實現真正的跨組織互通。 |
| **Service boundaries** | 代理可以存在於不同的微服務、雲端區域，甚至不同的組織，仍然能無縫協作。 |
| **Dynamic discovery** | 協調器可以在運行時查詢 Agent Card 登記庫，以尋找最適合處理特定子任務的專家。 |
| **Streaming & push notifications** | A2A 支援 Server-Sent Events (SSE) 以提供即時進度更新，並為長時間運行的任務提供推送通知。 |

我們上面建立的工作流程是這個模式的一個簡化、進程內版本。在實際部署中，每個代理會暴露一個 HTTP 端點、發佈 Agent Card，並透過 A2A JSON-RPC 協議進行通訊。


## 總結

在本課你已學到:

1. **A2A 協議是甚麼** — 一個用於代理之間發現、訊息傳遞,
   和任務管理。
2. **如何建立專門化代理** — 一個貨幣兌換代理, 一個活動策劃代理,
   以及一個旅行管理協調者。
3. **如何將代理接入工作流程** — 使用 `WorkflowBuilder` 來模擬順序
   訊息在代理之間的傳遞。
4. **A2A 在生產環境中的運作方式** — 使跨框架、跨服務的協作成為可能
   並具動態發現和串流更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件乃使用 AI 翻譯服務 Co‑op Translator (https://github.com/Azure/co-op-translator) 所翻譯。雖然我們已盡力確保準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為具權威性的版本。對於重要或關鍵資訊，建議採用專業人工翻譯。我們不會對因使用本翻譯而引致的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
